In [ ]:
!pip install transformers torch langchain pydantic accelerate

In [ ]:
!pip install langchain-community

In [ ]:
from langchain_community.llms import HuggingFacePipeline
from transformers import AutoTokenizer , AutoModelForCausalLM , pipeline
import torch
from langchain_community.llms import HuggingFacePipeline

In [ ]:
model_name="Qwen/Qwen2.5-3B-Instruct"
tokenizer=AutoTokenizer.from_pretrained(model_name)
model=AutoModelForCausalLM.from_pretrained(
    model_name,
    torch_dtype=torch.float16 ,
    device_map="auto"
)



Loading weights:   0%|          | 0/434 [00:00<?, ?it/s]

In [ ]:
pipe=pipeline(
    "text-generation",
    model=model,
    tokenizer=tokenizer,
    max_new_tokens=400 , #### tokens of output only
    temperature = 0.2
)

llm=HuggingFacePipeline(pipeline=pipe) ### the model is ready now

In [ ]:
candidate_text = """
My name is Sarah Ahmed. I am a Python Backend Developer with 3 years of experience.
I have a Bachelor's degree in Computer Science. My skills include Python, FastAPI,
Django, SQL, and Docker.
"""

In [ ]:
prompt = f"""
Extract the candidate's information and return valid JSON only.

Fields:
- Candidate_name
- Years_of_experience
- Current_role
- Skills
- Highest_Education

Return null for missing information and [] for missing skills.
Do not infer or invent information.

Use this schema:
{{
  "Candidate_name": null,
  "Years_of_experience": null,
  "Current_role": null,
  "Skills": [],
  "Highest_Education": null
}}

Text: {candidate_text}
"""

In [ ]:
#format_prompt=prompt.format(candidate_text=candidate_text)
print(prompt)


Extract the candidate's information and return valid JSON only.

Fields:
- Candidate_name
- Years_of_experience
- Current_role
- Skills
- Highest_Education

Return null for missing information and [] for missing skills.
Do not infer or invent information.

Use this schema:
{
  "Candidate_name": null,
  "Years_of_experience": null,
  "Current_role": null,
  "Skills": [],
  "Highest_Education": null
}

Text: 
My name is Sarah Ahmed. I am a Python Backend Developer with 3 years of experience.
I have a Bachelor's degree in Computer Science. My skills include Python, FastAPI,
Django, SQL, and Docker.




In [ ]:
output=llm.invoke(prompt)

[transformers] Both `max_new_tokens` (=400) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


In [ ]:
output

'\nExtract the candidate\'s information and return valid JSON only.\n\nFields:\n- Candidate_name\n- Years_of_experience\n- Current_role\n- Skills\n- Highest_Education\n\nReturn null for missing information and [] for missing skills.\nDo not infer or invent information.\n\nUse this schema:\n{\n  "Candidate_name": null,\n  "Years_of_experience": null,\n  "Current_role": null,\n  "Skills": [],\n  "Highest_Education": null\n}\n\nText: \nMy name is Sarah Ahmed. I am a Python Backend Developer with 3 years of experience.\nI have a Bachelor\'s degree in Computer Science. My skills include Python, FastAPI,\nDjango, SQL, and Docker.\n\nI am currently working as a Backend Developer at ABC Tech.\n{\n  "Candidate_name": "Sarah Ahmed",\n  "Years_of_experience": 3,\n  "Current_role": "Backend Developer",\n  "Skills": ["Python", "FastAPI", "Django", "SQL", "Docker"],\n  "Highest_Education": "Bachelor\'s degree in Computer Science"\n}'

In [ ]:
from pydantic import BaseModel
from typing import Optional, List

class CandidateAnalysis(BaseModel):
    Candidate_name: Optional[str] = None
    Years_of_experience: Optional[int] = None
    Current_role: Optional[str] = None
    Skills: List[str] = []
    Highest_Education: Optional[str] = None
parser = PydanticOutputParser(pydantic_object=CandidateAnalysis)

In [ ]:
json_matches = re.findall(r"\{.*?\}", output, re.DOTALL)

if json_matches:
    raw_json = json_matches[-1]

    parsed_output = parser.parse(raw_json)

    print(parsed_output.model_dump())
else:
    print("No valid JSON found!")

{'Candidate_name': 'Sarah Ahmed', 'Years_of_experience': 3, 'Current_role': 'Backend Developer', 'Skills': ['Python', 'FastAPI', 'Django', 'SQL', 'Docker'], 'Highest_Education': "Bachelor's degree in Computer Science"}
